# BF520 DMS Functional Score Pipeline

Runs each pipeline step by importing its module. Each step saves its own output files.

| Step | Module | Input | Output |
|------|--------|-------|--------|
| 1 | `CleanPairer` | `variant_counts/` | `functional_selections_clean.csv` |
| 2 | `VariantMerger` | `functional_selections_clean.csv` + `variant_counts/` | `merged_output/*_merged.csv` |
| 3 | `MutationMapper` | `merged_output/` + site numbering map | `mapped_output/*_merged_mapped.csv` |
| 4 | `FunctionalScoreCalculator` | `mapped_output/` | `func_scores_output/*_merged_mapped_func_score.csv` |

## Configuration
All paths are loaded from `config.yaml` — edit there to change any path.

In [2]:
import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)

cfg = config["pipeline"]

VARIANT_COUNTS_DIR    = cfg["variant_counts_dir"]
SITE_NUMBERING_MAP    = cfg["site_numbering_map"]
FUNCTIONAL_SELECTIONS = cfg["functional_selections_clean"]
MERGED_OUTPUT_DIR     = cfg["merged_output_dir"]
MAPPED_OUTPUT_DIR     = cfg["mapped_output_dir"]
FUNC_SCORES_DIR       = cfg["func_scores_dir"]

print("variant_counts_dir   :", VARIANT_COUNTS_DIR)
print("site_numbering_map   :", SITE_NUMBERING_MAP)
print("functional_selections:", FUNCTIONAL_SELECTIONS)
print("merged_output_dir    :", MERGED_OUTPUT_DIR)
print("mapped_output_dir    :", MAPPED_OUTPUT_DIR)
print("func_scores_dir      :", FUNC_SCORES_DIR)

variant_counts_dir   : variant_counts
site_numbering_map   : site_numbering/site_numbering_map.csv
functional_selections: results/functional_selections_clean.csv
merged_output_dir    : results/merged_output
mapped_output_dir    : results/mapped_output
func_scores_dir      : results/func_scores_output


---
## Step 1 — Generate selection pairs
Scans `variant_counts/` and pairs each `VSVG_control` file with its matching `no-antibody_control` file by prefix, date, rescue batch, and replicate.

**Output:** `results/functional_selections_clean.csv`

In [2]:
from CleanPairer import CleanPairer

pairer = CleanPairer(
    variant_counts_dir=VARIANT_COUNTS_DIR,
    output_csv=FUNCTIONAL_SELECTIONS,
)
selections = pairer.run()
selections

Found 11 valid functional pairs
Saved: results/functional_selections_clean.csv


,preselection_sample,library,virus_batch,replicate,postselection_sample,preselection_library_sample,postselection_library_sample,selection_name
0,2022-07-20_rescue-2_VSVG_control_1,A,rescue-2,1,2022-07-20_rescue-2_no-antibody_control_1,A_2022-07-20_rescue-2_VSVG_control_1,A_2022-07-20_rescue-2_no-antibody_control_1,A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...
1,2022-07-20_rescue-2_VSVG_control_2,A,rescue-2,2,2022-07-20_rescue-2_no-antibody_control_2,A_2022-07-20_rescue-2_VSVG_control_2,A_2022-07-20_rescue-2_no-antibody_control_2,A_2022-07-20_rescue-2_VSVG_control_2_vs_2022-0...
2,2022-08-04_rescue-1_VSVG_control_1,A,rescue-1,1,2022-08-04_rescue-1_no-antibody_control_1,A_2022-08-04_rescue-1_VSVG_control_1,A_2022-08-04_rescue-1_no-antibody_control_1,A_2022-08-04_rescue-1_VSVG_control_1_vs_2022-0...
3,2022-08-04_rescue-1_VSVG_control_2,A,rescue-1,2,2022-08-04_rescue-1_no-antibody_control_2,A_2022-08-04_rescue-1_VSVG_control_2,A_2022-08-04_rescue-1_no-antibody_control_2,A_2022-08-04_rescue-1_VSVG_control_2_vs_2022-0...
4,2022-09-01_rescue-3_VSVG_control_1,A,rescue-3,1,2022-09-01_rescue-3_no-antibody_control_1,A_2022-09-01_rescue-3_VSVG_control_1,A_2022-09-01_rescue-3_no-antibody_control_1,A_2022-09-01_rescue-3_VSVG_control_1_vs_2022-0...
5,2022-09-27_rescue-3_VSVG_control_1,A,rescue-3,1,2022-09-27_rescue-3_no-antibody_control_1,A_2022-09-27_rescue-3_VSVG_control_1,A_2022-09-27_rescue-3_no-antibody_control_1,A_2022-09-27_rescue-3_VSVG_control_1_vs_2022-0...
6,2022-10-17_rescue-4_VSVG_control_1,A,rescue-4,1,2022-10-17_rescue-4_no-antibody_control_1,A_2022-10-17_rescue-4_VSVG_control_1,A_2022-10-17_rescue-4_no-antibody_control_1,A_2022-10-17_rescue-4_VSVG_control_1_vs_2022-1...
7,2022-08-04_rescue-2_VSVG_control_1,B,rescue-2,1,2022-08-04_rescue-2_no-antibody_control_1,B_2022-08-04_rescue-2_VSVG_control_1,B_2022-08-04_rescue-2_no-antibody_control_1,B_2022-08-04_rescue-2_VSVG_control_1_vs_2022-0...
8,2022-08-04_rescue-2_VSVG_control_2,B,rescue-2,2,2022-08-04_rescue-2_no-antibody_control_2,B_2022-08-04_rescue-2_VSVG_control_2,B_2022-08-04_rescue-2_no-antibody_control_2,B_2022-08-04_rescue-2_VSVG_control_2_vs_2022-0...
9,2022-09-12_rescue-1_VSVG_control_1,B,rescue-1,1,2022-09-12_rescue-1_no-antibody_control_1,B_2022-09-12_rescue-1_VSVG_control_1,B_2022-09-12_rescue-1_no-antibody_control_1,B_2022-09-12_rescue-1_VSVG_control_1_vs_2022-0...


---
## Step 2 — Merge pre- and post-selection variant counts
For each selection pair, outer-joins the pre- and post-selection variant count CSVs on barcode, fills missing counts with 0.

**Output:** `results/merged_output/*_merged.csv` (one file per selection)

In [3]:
from variantMerger import VariantMerger

merger = VariantMerger(
    variant_count_dir=VARIANT_COUNTS_DIR,
    selection_file=FUNCTIONAL_SELECTIONS,
    output_dir=MERGED_OUTPUT_DIR,
)
merger.run_all()

Variant count dir: variant_counts
Selection file: results/functional_selections_clean.csv
Output dir: results/merged_output

Processing: A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-07-20_rescue-2_no-antibody_control_1
Pre path: variant_counts/A_2022-07-20_rescue-2_VSVG_control_1.csv
Post path: variant_counts/A_2022-07-20_rescue-2_no-antibody_control_1.csv

Processing: A_2022-07-20_rescue-2_VSVG_control_2_vs_2022-07-20_rescue-2_no-antibody_control_2
Pre path: variant_counts/A_2022-07-20_rescue-2_VSVG_control_2.csv
Post path: variant_counts/A_2022-07-20_rescue-2_no-antibody_control_2.csv

Processing: A_2022-08-04_rescue-1_VSVG_control_1_vs_2022-08-04_rescue-1_no-antibody_control_1
Pre path: variant_counts/A_2022-08-04_rescue-1_VSVG_control_1.csv
Post path: variant_counts/A_2022-08-04_rescue-1_no-antibody_control_1.csv

Processing: A_2022-08-04_rescue-1_VSVG_control_2_vs_2022-08-04_rescue-1_no-antibody_control_2
Pre path: variant_counts/A_2022-08-04_rescue-1_VSVG_control_2.csv
Post path:

{'A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-07-20_rescue-2_no-antibody_control_1':                                           selection_name           barcode  \
 0      A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...  AAAAAAAAAAACCACA   
 1      A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...  AAAAAAAAAATAGCAG   
 2      A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...  AAAAAAAAAATCATAC   
 3      A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...  AAAAAAAAAATTATTA   
 4      A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...  AAAAAAAAACCGCAAA   
 ...                                                  ...               ...   
 44746  A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...  TTTTTGGATCTCACAA   
 44747  A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...  TTTTTGGCGGCGCCGC   
 44748  A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...  TTTTTTATAGACGATT   
 44749  A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...  TTTTTTCACAAGCCTA   
 44750  A_2022-07-20_rescue-2_VSVG_control_1_v

---
## Step 3 — Remap mutation positions to HXB2 reference numbering
Translates sequential amino acid positions to HXB2 reference numbering using the site numbering map. Adds `aa_substitutions_reference`, `n_aa_substitutions`, and `n_codon_substitutions` columns.

**Output:** `results/mapped_output/*_merged_mapped.csv` (one file per selection)

In [4]:
from MutationMapper import MutationMapper

mapper = MutationMapper(
    input_folder=MERGED_OUTPUT_DIR,
    map_file=SITE_NUMBERING_MAP,
    output_dir=MAPPED_OUTPUT_DIR,
)
mapper.run_all()

INFO: Initializing MutationMapper...
INFO: Found 11 files to process
INFO: Processing file: results/merged_output/A_2022-07-20_rescue-2_VSVG_control_2_vs_2022-07-20_rescue-2_no-antibody_control_2_merged.csv
INFO: Saved: results/mapped_output/A_2022-07-20_rescue-2_VSVG_control_2_vs_2022-07-20_rescue-2_no-antibody_control_2_merged_mapped.csv
INFO: Processing file: results/merged_output/A_2022-09-01_rescue-3_VSVG_control_1_vs_2022-09-01_rescue-3_no-antibody_control_1_merged.csv
INFO: Saved: results/mapped_output/A_2022-09-01_rescue-3_VSVG_control_1_vs_2022-09-01_rescue-3_no-antibody_control_1_merged_mapped.csv
INFO: Processing file: results/merged_output/A_2022-08-04_rescue-1_VSVG_control_1_vs_2022-08-04_rescue-1_no-antibody_control_1_merged.csv
INFO: Saved: results/mapped_output/A_2022-08-04_rescue-1_VSVG_control_1_vs_2022-08-04_rescue-1_no-antibody_control_1_merged_mapped.csv
INFO: Processing file: results/merged_output/B_2022-08-04_rescue-2_VSVG_control_1_vs_2022-08-04_rescue-2_no-anti

---
## Step 4 — Compute functional scores
For each mapped file, computes log2 enrichment scores relative to the wildtype (variants with 0 codon substitutions). Adds `func_score`, `func_score_var`, `pre_count_wt`, `post_count_wt`, and `library` columns.

**Output:** `results/func_scores_output/*_merged_mapped_func_score.csv` (one file per selection)

In [3]:
from FunctionalCalculator import FunctionalScoreCalculator

calc = FunctionalScoreCalculator(
    pseudocount=cfg["pseudocount"],
    min_preselection_counts=cfg["min_preselection_counts"],
    min_preselection_frac=cfg["min_preselection_frac"],
)
calc.run_all(
    input_dir=MAPPED_OUTPUT_DIR,
    output_dir=FUNC_SCORES_DIR,
    selections_file=FUNCTIONAL_SELECTIONS,
)


INFO: Found 11 files to process
INFO: Processing: A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-07-20_rescue-2_no-antibody_control_1_merged_mapped.csv
INFO: Starting Functional Score Calculation...
INFO: Computing WT counts...
INFO: WT rows found: 3990
INFO: WT computed for 1 selections
INFO: Dynamic pre-count threshold: 25
INFO: Computing functional scores...
INFO: Functional score calculation complete
INFO: Saved: results/func_scores_output/A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-07-20_rescue-2_no-antibody_control_1_merged_mapped_func_score.csv
INFO: Processing: A_2022-07-20_rescue-2_VSVG_control_2_vs_2022-07-20_rescue-2_no-antibody_control_2_merged_mapped.csv
INFO: Starting Functional Score Calculation...
INFO: Computing WT counts...
INFO: WT rows found: 3990
INFO: WT computed for 1 selections
INFO: Dynamic pre-count threshold: 24
INFO: Computing functional scores...
INFO: Functional score calculation complete
INFO: Saved: results/func_scores_output/A_2022-07-20_rescue-2_VSVG_co